In [ ]:
from IPython.display import clear_output
!pip install ragas
!pip install pymupdf
clear_output()

In [ ]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Masukkan OpenAI API key: ")

Masukkan OpenAI API key: ··········


In [ ]:
import httpx

doc_url = "https://is3.cloudhost.id/omniflow-dev/business-intelligence.pdf"
output_file = "business-intelligence.pdf"

with httpx.Client(timeout=60.0) as client:
    response = client.get(doc_url)
    response.raise_for_status()  # biar error kalau gagal
    with open(output_file, "wb") as f:
        f.write(response.content)

print("PDF berhasil disimpan:", output_file)

PDF berhasil disimpan: business-intelligence.pdf


In [ ]:
import fitz  # PyMuPDF

pdf_path = "business-intelligence.pdf"

# Buka dokumen
doc = fitz.open(pdf_path)

# Ambil seluruh teks dari semua halaman
all_text = []

for page in doc:
    text = page.get_text("text")
    if text:
        all_text.append(text)

# Gabungkan jadi satu string
contexts = "\n".join(all_text)

doc.close()

print("Berhasil baca PDF.")
print("Panjang contexts:", len(contexts))


Berhasil baca PDF.
Panjang contexts: 6499


In [ ]:
contexts

'Business Intelligence: What It Is, How It Works, Its Importance,\nExamples, & Tools\nBusiness intelligence can help companies make better decisions by showing present\nand historical data within their business context. Analysts can leverage BI to provide\nperformance and competitor benchmarks to make the organization run smoother\nand more efficiently. Analysts can also more easily spot market trends to increase\nsales or revenue. Used effectively, the right data can help with anything from\ncompliance to hiring efforts. A few ways that business intelligence can help\ncompanies make smarter, data-driven decisions:\n-\nIdentify ways to increase profit\n-\nAnalyze customer behavior\n-\nCompare data with competitors\n-\nTrack performance\n-\nOptimize operations\n-\nPredict success\n-\nSpot market trends\n-\nDiscover issues or problems\nHow does it work? We can refer to the most popular framework for executing Data\nScience projects : CRISP-DM.\n\nThe CRoss Industry Standard Process for D

In [ ]:
import getpass
import os

os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("Masukkan API KEY DeepSeek")

Masukkan API KEY DeepSeek··········


In [ ]:
from openai import OpenAI
import os

generator_llm = OpenAI(
    api_key=os.environ.get("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

question = input("Masukkan pertanyaan Anda:\n")

response = generator_llm.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant that answers based only on the provided context."
        },
        {
            "role": "user",
            "content": f"""
Berikut adalah isi dokumen:

{contexts}

Pertanyaan:
{question}
"""
        }
    ],
    stream=False
)

print("\n")
print("===" * 50)
print(response.choices[0].message.content)

Masukkan pertanyaan Anda:
apa itu BI?


Berdasarkan dokumen yang diberikan, **BI (Business Intelligence)** adalah suatu pendekatan yang membantu perusahaan membuat keputusan yang lebih baik dengan menampilkan data saat ini dan historis dalam konteks bisnis mereka. BI memungkinkan analis untuk memberikan tolok ukur kinerja dan pesaing agar organisasi berjalan lebih lancar dan efisien, serta lebih mudah mengidentifikasi tren pasar untuk meningkatkan penjualan atau pendapatan. Data yang tepat dapat digunakan untuk berbagai hal mulai dari kepatuhan hingga upaya perekrutan.


In [ ]:
os.environ["GEMINI_API_KEY"] = getpass.getpass("Masukkan Gemini API Key:\n")

KeyboardInterrupt: Interrupted by user

In [ ]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecision, ContextRecall, ContextEntityRecall, Faithfulness, NoiseSensitivity, AnswerRelevancy

# client = AsyncOpenAI(
#     base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
#     api_key=os.environ["GEMINI_API_KEY"],
# )
client = AsyncOpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
)

# evaluator_llm = llm_factory("gemini-3-flash-preview", client=client)
evaluator_llm = llm_factory("gpt-5-mini", client=client, max_tokens=4096)

In [ ]:
metrics = {
    "faithfulness": Faithfulness,
    "context_precision": ContextPrecision,
    "context_recall": ContextRecall,
    "context_entity_recall": ContextEntityRecall,
    "noise_sensitivity": NoiseSensitivity,
    "answer_relevancy": AnswerRelevancy,
}

metric_name = "faithfulness"  # ganti sesuai kebutuhan

scorer = metrics[metric_name](llm=evaluator_llm)

MAX_CHARS = 8000
trimmed_context = contexts[:MAX_CHARS]

if metric_name == "faithfulness":
    result = await scorer.ascore(
        user_input=question,
        response=response.choices[0].message.content,
        retrieved_contexts=[trimmed_context]
    )
else:
    result = await scorer.ascore(
        user_input=question,
        reference=response.choices[0].message.content,
        retrieved_contexts=[trimmed_context]
    )

print(f"{metric_name.upper()} Score: {result.value}")


FAITHFULNESS Score: 0.9
